In [1]:
import pandas as pd
import numpy as np
import random as rd
from sklearn.model_selection import train_test_split

In [2]:
dados = pd.read_csv('dados_renda_municipios.csv',
                    sep = ',',
                    decimal= '.',
                    encoding = 'utf-8')

In [3]:
dados.head()

,UF,Municipio,RDPC
0,Rondônia,ALTA FLORESTA D'OESTE,476.99
1,Rondônia,ARIQUEMES,689.95
2,Rondônia,CABIXI,457.17
3,Rondônia,CACOAL,738.06
4,Rondônia,CEREJEIRAS,577.18


In [ ]:
dados.shape

(5565, 3)

## Amostragem por estado

- Selecionar o estado

In [6]:
# ajustes iniciais 
uf = 'São Paulo'

In [8]:
dados_municipio = dados[dados['UF'] == uf].reset_index(drop=True)

In [9]:
dados_municipio.shape

(645, 3)

- Ciração de estratos

In [ ]:
# Definição de 4 estratos, onde dividimos a população por quartis baseado em um critério intervalar
dados_municipio['classe_renda'] = pd.qcut(dados_municipio['RDPC'], 4, labels = ['D', 'C', 'B', 'A'])

In [11]:
dados_municipio

,UF,Municipio,RDPC,classe_renda
0,São Paulo,ADAMANTINA,975.43,A
1,São Paulo,ADOLFO,661.65,C
2,São Paulo,AGUAÍ,636.07,C
3,São Paulo,ÁGUAS DA PRATA,853.39,A
4,São Paulo,ÁGUAS DE LINDÓIA,730.13,B
...,...,...,...,...
640,São Paulo,VOTORANTIM,703.99,B
641,São Paulo,VOTUPORANGA,977.39,A
642,São Paulo,ZACARIAS,605.79,C
643,São Paulo,CHAVANTES,677.30,C


In [ ]:
# dados gerais amostra piloto 
dados_piloto = dados_municipio.agg( media_RDPC = pd.NamedAgg('RDPC', 'mean'),
                                    dp_RDPC = pd.NamedAgg('RDPC', 'std'),
                                    N = pd.NamedAgg('RDPC', 'count'))
dados_piloto

,RDPC
media_RDPC,713.926155
dp_RDPC,197.398270
N,645.000000


In [ ]:
# dados piloto por estrato
dados_piloto_classe = dados_municipio.groupby('classe_renda').agg( media_RDPC = pd.NamedAgg('RDPC', 'mean'),
                                                                    dp_RDPC = pd.NamedAgg('RDPC', 'std'),
                                                                    N = pd.NamedAgg('RDPC', 'count')) .reset_index()

dados_piloto_classe['percent'] = dados_piloto_classe['N'] / sum(dados_piloto_classe['N'])
dados_piloto_classe

C:\Users\luizf\AppData\Local\Temp\ipykernel_24120\3410766707.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dados_piloto_classe = dados_municipio.groupby('classe_renda') .agg(media_RDPC = pd.NamedAgg('RDPC', 'mean'),


,classe_renda,media_RDPC,dp_RDPC,N,percent
0,D,517.531296,58.889964,162,0.251163
1,C,637.659259,28.839032,162,0.251163
2,B,736.007313,34.142763,160,0.248062
3,A,966.337453,204.484141,161,0.249612


In [16]:
# formula continua
def formula_amostra_continua(N, S, Z, ME):
    n = (Z**2 * S**2 * N) / ((ME**2 * (N-1)) + (Z**2 * S**2))
    return int(n)

In [15]:
# formula discreta
def formula_amostra_discreta(N, Z, ME):
    n = (Z**2 * 0.25 * N) / ((ME**2 * (N-1)) + (Z**2 * 0.25))
    return int(n)

In [18]:
# parametros
N = 645
Z = 1.96
S = 197.40
ME = 25
# tamanho amostra
n = formula_amostra_continua(N, S, Z, ME)
n

174

## Amostra aleatória simples

- Função Random

In [20]:
# sorteio das linhas
linhas_sorteadas = rd.sample(range(1, N+1), n)

In [21]:
# filtrar os dados
dados_amostra = dados_municipio[dados_municipio.index.isin(linhas_sorteadas)]
dados_amostra.shape

(173, 4)

In [22]:
dados_amostra

,UF,Municipio,RDPC,classe_renda
4,São Paulo,ÁGUAS DE LINDÓIA,730.13,B
12,São Paulo,ALTO ALEGRE,560.30,D
15,São Paulo,ÁLVARES MACHADO,672.03,C
16,São Paulo,ÁLVARO DE CARVALHO,512.99,D
17,São Paulo,ALVINLÂNDIA,710.49,B
...,...,...,...,...
622,São Paulo,UBIRAJARA,580.57,D
626,São Paulo,URU,544.10,D
632,São Paulo,VARGEM GRANDE DO SUL,711.60,B
639,São Paulo,VITÓRIA BRASIL,572.68,D


- Função Sample

In [23]:
dados_amostra_simples = dados_municipio.sample(n=174)
dados_amostra_simples.shape

(174, 4)

In [24]:
dados_amostra_simples

,UF,Municipio,RDPC,classe_renda
368,São Paulo,NOVA CAMPINA,330.03,D
309,São Paulo,LUCÉLIA,743.20,B
125,São Paulo,CASTILHO,579.36,D
233,São Paulo,ILHA COMPRIDA,608.83,C
595,São Paulo,TAPIRAÍ,449.75,D
...,...,...,...,...
315,São Paulo,MACATUBA,920.95,A
177,São Paulo,FERNANDÓPOLIS,944.00,A
122,São Paulo,CARDOSO,605.99,C
483,São Paulo,RIBEIRÃO DO SUL,866.43,A


- Amostra Estratificada

In [ ]:
# amostra estratificada
dados_amostra_estrat = train_test_split(dados_municipio, 
                                        test_size = n, 
                                        random_state=1245, 
                                        stratify=dados_municipio['classe_renda'])[1]
dados_amostra_estrat.shape

(174, 4)

In [ ]:
# resumo amostra estratificada
resumo_estrat = dados_amostra_estrat.groupby('classe_renda').agg(media_RDPC = pd.NamedAgg('RDPC', 'mean'),
                                                                dp_RDPC = pd.NamedAgg('RDPC', 'std'),
                                                                N = pd.NamedAgg('RDPC', 'count')) .reset_index()

resumo_estrat['percent'] = resumo_estrat['N'] / sum(resumo_estrat['N'])
resumo_estrat

C:\Users\luizf\AppData\Local\Temp\ipykernel_24120\2686331823.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  resumo_estrat = dados_amostra_estrat.groupby('classe_renda') .agg(media_RDPC = pd.NamedAgg('RDPC', 'mean'),


,classe_renda,media_RDPC,dp_RDPC,N,percent
0,D,530.192500,42.159454,44,0.252874
1,C,635.421591,28.204496,44,0.252874
2,B,733.693953,34.054255,43,0.247126
3,A,982.552558,232.962593,43,0.247126


In [27]:
dados_piloto

,RDPC
media_RDPC,713.926155
dp_RDPC,197.398270
N,645.000000


In [28]:
dados_piloto_classe

,classe_renda,media_RDPC,dp_RDPC,N,percent
0,D,517.531296,58.889964,162,0.251163
1,C,637.659259,28.839032,162,0.251163
2,B,736.007313,34.142763,160,0.248062
3,A,966.337453,204.484141,161,0.249612


- Amostra aleatória simples (calculando adiferença)

In [30]:

dados_piloto_classe = dados_municipio.agg( media_RDPC = pd.NamedAgg('RDPC', 'mean'),
                                    dp_RDPC = pd.NamedAgg('RDPC', 'std'),
                                    N = pd.NamedAgg('RDPC', 'count'))
dados_piloto_classe

,RDPC
media_RDPC,713.926155
dp_RDPC,197.398270
N,645.000000
